In [1]:
!pip install -q evaluate==0.4.1
!pip install -q jiwer==3.0.3
!pip install -q librosa soundfile
!pip install -q faster-whisper   

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 79.1 MB/s eta 0:00:00


In [2]:
!pip install -U bitsandbytes>=0.46.1

In [3]:
import os
import re
import gc
import time
import warnings
import dataclasses
from pathlib import Path
from typing import Any, Dict, List, Union

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
from tqdm.notebook import tqdm


from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from transformers import BitsAndBytesConfig
from datasets import Dataset, Audio
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import evaluate
from jiwer import wer as compute_wer

warnings.filterwarnings("ignore")


In [4]:
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"GPU count        : {n_gpus}")
    for i in range(n_gpus):
        name = torch.cuda.get_device_name(i)
        mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {name} ({mem:.1f} GB)")
    DEVICE = "cuda"
    N_GPUS = n_gpus
else:
    DEVICE = "cpu"
    N_GPUS = 0
    print("No GPU found!")

print(f"Active device: {DEVICE}")

PyTorch version  : 2.10.0+cu128
CUDA available   : True
GPU count        : 2
  GPU 0: Tesla T4 (15.6 GB)
  GPU 1: Tesla T4 (15.6 GB)
Active device: cuda


In [5]:
class CFG:
    TRAIN_CSV       = Path("/kaggle/input/competitions/multilingual-speech-recognition/train.csv")
    TEST_CSV        = Path("/kaggle/input/competitions/multilingual-speech-recognition/test.csv")
    SAMPLE_SUB      = Path("/kaggle/input/competitions/multilingual-speech-recognition/sample_submission.csv")
    TRAIN_AUDIO_DIR = Path("/kaggle/input/competitions/multilingual-speech-recognition/competition_data/train")
    TEST_AUDIO_DIR  = Path("/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test")
    OUTPUT_DIR      = Path("/kaggle/working")
    FT_MODEL_DIR    = Path("/kaggle/working/whisper-qlora-ft")
    MERGED_DIR      = Path("/kaggle/working/whisper-qlora-merged")

MODEL_ID        = "openai/whisper-medium"
TARGET_SR       = 16000
MAX_DURATION_S  = 30
MAX_LABEL_LENGTH= 448
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"


LORA_R          = 8
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.1
LORA_TARGETS    = ["q_proj", "v_proj"]


TRAIN_VAL_SPLIT = 0.1
SEED            = 42
TRAIN_BS        = 2
EVAL_BS         = 4
GRAD_ACCUM      = 8
MAX_STEPS       = 1000
WARMUP_STEPS    = 200
LR              = 2e-5
EVAL_STEPS      = 100
SAVE_STEPS      = 100
LOG_STEPS       = 25


BEAM_SIZE       = 5

In [6]:
train_df  = pd.read_csv(CFG.TRAIN_CSV)
test_df   = pd.read_csv(CFG.TEST_CSV)
sample_df = pd.read_csv(CFG.SAMPLE_SUB)

In [7]:

display(train_df.head(3))

,id,audio,text
0,0,audio_00000.wav,you had quoted plutarch line.
1,1,audio_00001.wav,மலையேறுதலில் வந்து பார்த்தீங்கன்னா ஜஸ்ட்டு நம்...
2,2,audio_00002.wav,to do his phd in engineering about four years ...


In [8]:

display(test_df.head(3))

,id,audio
0,0,audio_00000.wav
1,1,audio_00001.wav
2,2,audio_00002.wav


In [9]:

display(sample_df.head(3))

,audio,text
0,audio_00000.wav,NaN
1,audio_00001.wav,NaN
2,audio_00002.wav,NaN


In [10]:
AUDIO_COL       = "audio"   
TRAIN_AUDIO_COL = "audio"   
TEXT_COL        = "text"    
SUB_AUDIO_COL   = "audio"   
SUB_TEXT_COL    = "text"    
LANG_COL        = None 

In [11]:
print(f"Train audio col : '{TRAIN_AUDIO_COL}'")
print(f"Train text col  : '{TEXT_COL}'")
print(f"Test audio col  : '{AUDIO_COL}'")

Train audio col : 'audio'
Train text col  : 'text'
Test audio col  : 'audio'


In [12]:
def load_audio(filepath):
    y, sr = librosa.load(str(filepath), sr=TARGET_SR, mono=True)
    y, _  = librosa.effects.trim(y, top_db=20)
    peak  = np.max(np.abs(y))
    if peak > 0:
        y = y / peak
    return y.astype(np.float32)

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Audio utils defined")

Audio utils defined


In [13]:
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    language=None,
    task="transcribe",
)

feature_extractor = processor.feature_extractor
tokenizer         = processor.tokenizer

print(f" Processor loaded")
print(f"   Vocab size : {tokenizer.vocab_size}")

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

 Processor loaded
   Vocab size : 50258


In [14]:
import random

data = []
for _, row in train_df.iterrows():
    data.append({
        "audio" : str(CFG.TRAIN_AUDIO_DIR / row["audio"]),
        "text"  : row["text"],
    })

random.seed(SEED)
random.shuffle(data)

n_val   = int(len(data) * TRAIN_VAL_SPLIT)
train_data = data[n_val:]
val_data   = data[:n_val]

print(f"Train : {len(train_data)}")
print(f"Val   : {len(val_data)}")


Train : 1800
Val   : 200


In [15]:
def preprocess_sample(sample):
    y = load_audio(sample["audio"])
    y = y[:TARGET_SR * MAX_DURATION_S]

    input_features = feature_extractor(
        y, sampling_rate=TARGET_SR, return_tensors="np"
    ).input_features[0]

    labels = tokenizer(
        sample["text"],
        max_length=MAX_LABEL_LENGTH,
        truncation=True,
        add_special_tokens=False,
    ).input_ids

    return {"input_features": input_features, "labels": labels}

In [16]:
hf_train = Dataset.from_list(train_data).map(
    preprocess_sample,
    remove_columns=["audio", "text"],
    desc="Preprocessing train",
)

hf_val = Dataset.from_list(val_data).map(
    preprocess_sample,
    remove_columns=["audio", "text"],
    desc="Preprocessing val",
)

Preprocessing train:   0%|          | 0/1800 [00:00<?, ? examples/s]

Preprocessing val:   0%|          | 0/200 [00:00<?, ? examples/s]

In [17]:
print(f"\ Datasets ready")
print(f"   Train : {len(hf_train)}")
print(f"   Val   : {len(hf_val)}")

\ Datasets ready
   Train : 1800
   Val   : 200


In [18]:
from peft import LoraConfig, get_peft_model, TaskType

In [19]:
base_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype = torch.float16,
    device_map          = "auto",
)


base_model.config.use_cache = False

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [20]:
lora_config = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    target_modules = LORA_TARGETS,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    task_type      = TaskType.SEQ_2_SEQ_LM,
)

In [21]:
model = get_peft_model(base_model, lora_config)
model.enable_input_require_grads()

In [22]:

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens    = []
model.generation_config.language           = None
model.generation_config.task               = "transcribe"

In [23]:
@dataclasses.dataclass
class WhisperDataCollator:
    processor: WhisperProcessor
    decoder_start_token_id: int

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        batch["input_features"] = batch["input_features"].to(torch.float16)  # ← here

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        bos_token_id = self.processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
        while labels.shape[1] > 0 and labels[:, 0].eq(bos_token_id).any():
            labels = labels[:, 1:]
        while labels.shape[1] > 0 and (labels[:, 0] > 50257).any():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

In [24]:
def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    predictions = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    references  = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    predictions = [normalize_text(p) for p in predictions]
    references  = [normalize_text(r) for r in references]

    return {"wer": round(compute_wer(references, predictions), 4)}



data_collator = WhisperDataCollator(
    processor              = processor,
    decoder_start_token_id = model.config.decoder_start_token_id,
)

In [25]:
training_args = Seq2SeqTrainingArguments(
    output_dir                  = str(CFG.FT_MODEL_DIR),
    per_device_train_batch_size = TRAIN_BS,
    per_device_eval_batch_size  = EVAL_BS,
    gradient_accumulation_steps = GRAD_ACCUM,
    max_steps                   = MAX_STEPS,
    warmup_steps                = WARMUP_STEPS,
    learning_rate               = LR,
    fp16                        = True,
    bf16                        = False,
    gradient_checkpointing      = True,
    optim                       = "adamw_torch",
    max_grad_norm               = 1.0,
    eval_strategy               = "steps",
    eval_steps                  = EVAL_STEPS,
    predict_with_generate       = True,
    generation_max_length       = MAX_LABEL_LENGTH,
    save_strategy               = "steps",
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "wer",
    greater_is_better           = False,
    logging_steps               = LOG_STEPS,
    report_to                   = "none",
    seed                        = SEED,
    remove_unused_columns       = False,
)

In [26]:
trainer = Seq2SeqTrainer(
    model             = model,
    args              = training_args,
    train_dataset     = hf_train,
    eval_dataset      = hf_val,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,
    processing_class  = processor.feature_extractor,
    callbacks         = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Trainer ready")


Trainer ready


In [27]:
trainer.train()

Step,Training Loss,Validation Loss,Wer
100,27.888865,2.390631,0.429800
200,12.104830,1.111908,0.440000
300,6.949969,0.792390,0.560300
400,6.543126,0.701512,0.676600


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

TrainOutput(global_step=400, training_loss=16.0225870513916, metrics={'train_runtime': 6540.1288, 'train_samples_per_second': 2.446, 'train_steps_per_second': 0.153, 'total_flos': 6.52903862501376e+18, 'train_loss': 16.0225870513916, 'epoch': 3.542222222222222})

In [28]:
# Save best checkpoint (step 200)
model.save_pretrained(str(CFG.FT_MODEL_DIR))
processor.save_pretrained(str(CFG.FT_MODEL_DIR))
print("Best model saved")

Best model saved


In [29]:
model.eval()
model.generation_config.forced_decoder_ids = None
model.generation_config.language           = None
model.generation_config.task               = "transcribe"

In [30]:
all_predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    fpath = CFG.TEST_AUDIO_DIR / row["audio"]
    
    try:
        y     = load_audio(fpath)
        y     = y[:TARGET_SR * MAX_DURATION_S]
        feats = processor(y, sampling_rate=TARGET_SR, return_tensors="pt")
        feats = feats.input_features.to(torch.float16).to(DEVICE)
        
        with torch.no_grad():
            ids = model.generate(
                input_features = feats,
                num_beams      = 5,
                max_new_tokens = 200,
            )
        
        text = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

        all_predictions.append(text)
        
    except Exception as e:
        print(f"{row['audio']}: {e}")
        all_predictions.append("")

Inference:   0%|          | 0/100 [00:00<?, ?it/s]

Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [31]:
submission = pd.DataFrame({
    "audio" : test_df["audio"].values,
    "text"  : all_predictions,
})


In [32]:
submission.to_csv("/kaggle/working/submission.csv", index=False)
display(submission.head(10))

,audio,text
0,audio_00000.wav,ان میں جی پی مورگن، سیمس اور بلیک روک کے سیئیو...
1,audio_00001.wav,एक हुजार नोसो नब्वे में जहां दो मुस्लिन मुमिद्...
2,audio_00002.wav,மாட்டுப்பொங்கள் முடிந்து மாட்டுப்பொங்கள் நைட்ட...
3,audio_00003.wav,அறிவிக்குற மாதிரி ஒரு படை இருக்கணும்
4,audio_00004.wav,And by this I knew the violence of my love. I ...
5,audio_00005.wav,"Really the beginning, as the event was to show..."
6,audio_00006.wav,I'm sure that is all you could expect from us....
7,audio_00007.wav,जहाँ तक सम्भो हो सीधी उडान ले
8,audio_00008.wav,சேர்க்கியான வெளிச்சத்தை நூக்கி தான் போறாங்க ஒர...
9,audio_00009.wav,हजार नो सूपें सट करोट रूपे का खर्च आया था
